# Induction Motor Tests

## Run C++ Example: Induction Motor Load Switch

In [ ]:
import subprocess
from pathlib import Path

name = "EMT_Ph3_SSN_InductionMotor_LoadSwitch"

dpsim_path = Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"])
    .decode("utf-8")
    .strip()
)
path_exec = dpsim_path / "build" / "dpsim" / "examples" / "cxx" / name

result = subprocess.run(
    [str(path_exec)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True
)
print(result.stdout.decode())

## Load Results

In [ ]:
from villas.dataprocessing.readtools import *
import matplotlib.pyplot as plt
import numpy as np

sim_name = "EMT_Ph3_SSN_InductionMotor_ColdStart_LoadSwitch"
path_logfile = Path("logs") / sim_name / f"{sim_name}.csv"
ts_motor = read_timeseries_dpsim(str(path_logfile))

## Plot Results

In [ ]:
def values(name):
    return np.asarray(ts_motor[name].values, dtype=float)


time = ts_motor["mechanical_speed_pu"].time
speed_pu = values("mechanical_speed_pu")
slip = values("slip")
torque_pu = values("mechanical_load_torque")  # nominal torque is 1 Nm in the example
active_power = values("electrical_power")
reactive_power = values("reactive_power")
voltage_mag = values("stator_voltage_magnitude")
current_mag = values("stator_current_magnitude")

voltage_base = np.nanmedian(voltage_mag[-max(1, len(voltage_mag) // 10) :])
current_base = np.nanmedian(current_mag[-max(1, len(current_mag) // 10) :])
voltage_pu = voltage_mag / voltage_base
current_pu = current_mag / current_base

plt.close("all")
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(time, active_power, label="Active power [W]")
axes[0, 0].plot(time, reactive_power, label="Reactive power [VAr]")
axes[0, 0].set_title("Active and reactive power")
axes[0, 0].set_xlabel("Time [s]")
axes[0, 0].grid(True)
axes[0, 0].legend()

axes[0, 1].plot(time, torque_pu, label="Mechanical load torque [p.u.]")
axes[0, 1].plot(time, speed_pu, label="Speed [p.u.]")
axes[0, 1].plot(time, slip, "--", label="Slip [p.u.]")
axes[0, 1].set_title("Mechanical torque, speed and slip")
axes[0, 1].set_xlabel("Time [s]")
axes[0, 1].grid(True)
axes[0, 1].legend()

axes[1, 0].plot(time, voltage_pu, label="Voltage magnitude [p.u.]")
axes[1, 0].plot(time, current_pu, "--", label="Stator current magnitude [p.u.]")
axes[1, 0].set_title("Voltage magnitude and stator current")
axes[1, 0].set_xlabel("Time [s]")
axes[1, 0].grid(True)
axes[1, 0].legend()

axes[1, 1].plot(speed_pu, active_power, label="Active power [W]")
axes[1, 1].plot(speed_pu, reactive_power, label="Reactive power [VAr]")
axes[1, 1].set_title("Active/reactive power vs. speed")
axes[1, 1].set_xlabel("Speed [p.u.]")
axes[1, 1].grid(True)
axes[1, 1].legend()

fig.tight_layout()

## Basic Checks

In [ ]:
for signal in [
    "mechanical_speed_pu",
    "slip",
    "electrical_power",
    "reactive_power",
    "mechanical_load_torque",
    "stator_voltage_magnitude",
    "stator_current_magnitude",
]:
    assert np.all(np.isfinite(values(signal))), f"{signal} contains non-finite values"

assert np.nanmax(speed_pu) > 0.5
assert np.nanmax(current_pu) > 1.0